In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/phi-2_adaptive-lora/results', exist_ok=True)
print("Drive ready")

Mounted at /content/drive
Drive ready


In [ ]:
%%capture
!pip install transformers peft accelerate bitsandbytes datasets wandb trl


In [ ]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from datasets import load_dataset

print("Imports done")

Imports done


In [ ]:
model_name = "microsoft/phi-2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

print(f"Loaded. Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
for name, module in model.named_modules():
    if any(x in name for x in ['proj', 'Wqkv', 'fc']):
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.mlp.fc1
model.layers.0.mlp.fc2
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.mlp.fc1
model.layers.1.mlp.fc2
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.mlp.fc1
model.layers.2.mlp.fc2
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.mlp.fc1
model.layers.3.mlp.fc2
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.mlp.fc1
model.layers.4.mlp.fc2
model.layers.5.self_attn.q_proj
model.layers.5.self_attn.k_proj
model.layers.5.self_attn.v_proj
model.layers.5.mlp.fc1
model.layers.5.mlp.fc2
model.layers.6.self_attn.q_proj
model.layers.6.self_attn.k_proj
model.layers.6.self_attn.v_proj
model.layers.6.mlp.fc1
model.layers.6.mlp.fc2
model.

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["Wqkv"],   # adjust based on what you saw printed
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 2,621,440 || all params: 2,782,305,280 || trainable%: 0.0942


In [ ]:
dataset = load_dataset("fancyzhx/ag_news")

def format_example(example):
    label_names = ['World', 'Sports', 'Business', 'Sci/Tech']
    label = label_names[example['label']]
    text = f"""Classify the following news article into one of these categories: World, Sports, Business, Sci/Tech.

Article: {example['text']}

Category: {label}"""
    return {"text": text}

formatted = dataset['train'].map(format_example)
print(formatted[0]['text'])

In [ ]:
MAX_LENGTH = 256

def tokenize(example):
    result = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

warmup_data = formatted.select(range(1000))
tokenized = warmup_data.map(tokenize, remove_columns=warmup_data.column_names)
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Tokenized {len(tokenized)} examples")

In [ ]:
from torch.utils.data import DataLoader

warmup_loader = DataLoader(tokenized, batch_size=4, shuffle=True)
print(f"Batches: {len(warmup_loader)}")

Batches: 250


In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4
)

sensitivity = {}
for name, param in model.named_parameters():
    if param.requires_grad:
        sensitivity[name] = 0.0

NUM_WARMUP_STEPS = 200

model.train()
step = 0

for batch in tqdm(warmup_loader):
    if step >= NUM_WARMUP_STEPS:
        break

    batch = {k: v.to(model.device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()

    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            sensitivity[name] += (param.grad.norm() / param.grad.numel() ** 0.5).item()

    optimizer.step()
    optimizer.zero_grad()
    step += 1

    if step % 50 == 0:
        print(f"Step {step}/{NUM_WARMUP_STEPS} | Loss: {loss.item():.4f}")

for name in sensitivity:
    sensitivity[name] /= NUM_WARMUP_STEPS

print("Done")

  0%|          | 0/250 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
 20%|██        | 50/250 [01:35<06:38,  1.99s/it]

Step 50/200 | Loss: 0.8670


 40%|████      | 100/250 [03:21<05:14,  2.10s/it]

Step 100/200 | Loss: 0.6094


 60%|██████    | 150/250 [05:07<03:30,  2.11s/it]

Step 150/200 | Loss: 0.5980


 80%|████████  | 200/250 [06:53<01:43,  2.07s/it]

Step 200/200 | Loss: 0.6791
Done


In [ ]:
save_path = '/content/drive/MyDrive/phi-2_adaptive-lora/sensitivity_scores.json'
with open(save_path, 'w') as f:
    json.dump(sensitivity, f, indent=2)

print(f"Saved to {save_path}")

sorted_layers = sorted(sensitivity.items(), key=lambda x: x[1], reverse=True)
print("\nTop 10 most sensitive:")
for name, score in sorted_layers[:10]:
    print(f"{score:.4f}  {name}")

print("\nBottom 10 least sensitive:")
for name, score in sorted_layers[-10:]:
    print(f"{score:.4f}  {name}")

Saved to /content/drive/MyDrive/phi-2_adaptive-lora/sensitivity_scores.json

Top 10 most sensitive:
0.0030  base_model.model.base_model.model.model.layers.29.self_attn.v_proj.lora_B.default.weight
0.0022  base_model.model.base_model.model.model.layers.29.self_attn.q_proj.lora_B.default.weight
0.0011  base_model.model.base_model.model.model.layers.2.self_attn.v_proj.lora_B.default.weight
0.0010  base_model.model.base_model.model.model.layers.4.self_attn.v_proj.lora_B.default.weight
0.0010  base_model.model.base_model.model.model.layers.30.self_attn.v_proj.lora_B.default.weight
0.0010  base_model.model.base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
0.0009  base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
0.0009  base_model.model.base_model.model.model.layers.3.self_attn.v_proj.lora_B.default.weight
0.0008  base_model.model.base_model.model.model.layers.5.self_attn.v_proj.lora_B.default.weight
0.0008  base_model.model.base_mod

In [ ]:
import json
import re

with open('/content/drive/MyDrive/phi-2_adaptive-lora/sensitivity_scores.json', 'r') as f:
    raw_sensitivity = json.load(f)

aggregated = {}
counts = {}

# adjust this regex based on what you printed above
for full_name, score in raw_sensitivity.items():
    match = re.search(r'layers\.(\d+)\.(?:self_attn|mixer)\.(Wqkv|q_proj|v_proj)', full_name)
    if match:
        layer_num = match.group(1)
        proj_type = match.group(2)
        key = f"layer_{layer_num}_{proj_type}"
        aggregated[key] = aggregated.get(key, 0.0) + score
        counts[key] = counts.get(key, 0) + 1

# average instead of sum — Bug 2 fix
aggregated = {k: v / counts[k] for k, v in aggregated.items()}

print(f"Aggregated into {len(aggregated)} layer-projection entries")

with open('/content/drive/MyDrive/phi-2_adaptive-lora/results/sensitivity_aggregated.json', 'w') as f:
    json.dump(aggregated, f, indent=2)

sorted_agg = sorted(aggregated.items(), key=lambda x: x[1], reverse=True)
print("\nTop 10 aggregated:")
for name, score in sorted_agg[:10]:
    print(f"{score:.4f}  {name}")

Aggregated into 64 layer-projection entries

Top 10 aggregated:
0.0019  layer_29_v_proj
0.0012  layer_29_q_proj
0.0007  layer_30_v_proj
0.0007  layer_0_v_proj
0.0007  layer_2_v_proj
0.0006  layer_4_v_proj
0.0006  layer_31_v_proj
0.0006  layer_1_v_proj
0.0005  layer_26_v_proj
0.0005  layer_3_v_proj
